In [ ]:
#@title Step ０: Install Required Libraries
#@markdown Run this cell to install necessary Python packages (e.g., Biopython) for this pipeline.

print("📦 Installing required packages... (This may take a few seconds)")

# The -q (quiet) flag hides verbose output for a cleaner UI
!pip install -q biopython

print("✅ Installation complete! ")

In [ ]:
#@title Step 1: Name your project
#@markdown Enter a name for your project. This name will be used to tag all output files generated in subsequent steps.
PROJECT_NAME = "" #@param {type:"string"}

print(f"✅ Project name successfully set to: '{PROJECT_NAME}'")
print("💡 All output files will be automatically tagged with this name.")

In [ ]:
#@title Step 2: siRNA Target Discovery
#@markdown Select whether to target a single gene or find common targets across multiple genes.

#@markdown > **Algorithm & Citation:**
#@markdown > - **Efficacy (Design Rules):** siRNA candidates are selected based on the guidelines by **Ui-Tei et al., *Nucleic Acids Res* 32, 936-948 (2004)**.
#@markdown > - **Specificity (Off-target Prevention):** To minimize seed-dependent off-target effects, the melting temperature (Tm) of the seed region (positions 2-8) is strictly kept at **21.5°C or lower**, following the thermodynamic stability criteria established by **Ui-Tei et al., *Nucleic Acids Res* 36, 7100-7109 (2008)**.

#@markdown ---
#@markdown #### **1. Targeting Mode**
TARGETING_MODE = "Common Target (Multiple Genes)" #@param ["Single Gene", "Common Target (Multiple Genes)"]
INPUT_METHOD = "Upload File" #@param ["Manual Text", "Upload File"]

#@markdown ---
#@markdown #### **2. Advanced Filter Settings**
#@markdown Optional: Restrict the 3' end of the target sequence to specific nucleotides.
#@markdown (Required for certain in vitro transcription kits, e.g., TaKaRa T7 siRNA Synthesis Kit).
RESTRICT_3PRIME_AU = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### **▼ IF [Single Gene] IS SELECTED ▼**
#@markdown *(Ignore this section if targeting multiple genes)*
SINGLE_GENE_NAME = "" #@param {type:"string"}
SINGLE_DNA_SEQUENCE = "" #@param {type:"string"}

#@markdown ---
#@markdown ### **▼ IF [Common Target] IS SELECTED ▼**
#@markdown *(Ignore this section if targeting a single gene)*
COMMON_FASTA_TEXT = "" #@param {type:"string"}

#@markdown ---
#@markdown #### **3. Evaluation Parameters**
TM_THRESHOLD = 21.5 #@param {type:"number"}
KMER_LEN = 23 #@param {type:"integer"}

import csv, sys, re, os, time, io
import pandas as pd
from google.colab import files

# --- 1. Library Setup ---
try:
    from Bio import SeqIO
    from Bio.Seq import Seq
    from Bio.SeqUtils import MeltingTemp as mt
except ImportError:
    print("📦 Installing Biopython...")
    !pip install -q biopython
    from Bio import SeqIO
    from Bio.Seq import Seq
    from Bio.SeqUtils import MeltingTemp as mt

# --- 2. siRNA Evaluation Logic ---
def evaluate_sirna_candidates(target_dna, tm_limit, restrict_3p):
    if len(target_dna) != 23: return None
    p_rna = target_dna[2:23].replace('T', 'U')
    g_rna = str(Seq(target_dna[0:21]).reverse_complement()).replace('T', 'U')

    # Ui-Tei Rules (2004)
    if not (g_rna[0] in 'AU' and p_rna[0] in 'GC' and sum(1 for b in g_rna[:7] if b in 'AU') >= 4): return None
    if re.search(r'[GC]{10,}', p_rna) or re.search(r'[GC]{10,}', g_rna): return None

    # Optional: 3' End Restriction (Check for A/U in the last 2 nucleotides of target)
    if restrict_3p:
        if not (all(b in 'AU' for b in p_rna[-2:]) and all(b in 'AU' for b in g_rna[-2:])): return None

    # Tm Calculation (Ref: Ui-Tei et al. 2008)
    tm_g = mt.Tm_NN(Seq(g_rna[1:8]), nn_table=mt.RNA_NN1, Na=100, dnac1=100000) - 12.6
    tm_p = mt.Tm_NN(Seq(p_rna[1:8]), nn_table=mt.RNA_NN1, Na=100, dnac1=100000) - 12.6

    if tm_g > tm_limit or tm_p > tm_limit: return None
    return {"g": g_rna, "p": p_rna, "tg": round(tm_g, 1), "tp": round(tm_p, 1)}

# --- 3. Main Process ---
p_name = PROJECT_NAME if 'PROJECT_NAME' in globals() else "Untitled_Project"
results = []
is_common_mode = "Common" in TARGETING_MODE
out_fname = f"Step2_{'Common' if is_common_mode else 'Single'}_Candidates_{p_name}.csv"

print(f"🚀 Running Step 2: {p_name} [{TARGETING_MODE}]")

# ==========================================
# Mode A: Single Gene
# ==========================================
if not is_common_mode:
    input_data = []
    if "Manual" in INPUT_METHOD:
        seq = SINGLE_DNA_SEQUENCE.strip().upper().replace('-', '').replace(' ', '')
        if len(seq) >= KMER_LEN:
            input_data.append((SINGLE_GENE_NAME, seq))
    else:
        uploaded = files.upload()
        if uploaded:
            fname = list(uploaded.keys())[0]
            raw_data = uploaded[fname]
            try: content = raw_data.decode('utf-8-sig')
            except: content = raw_data.decode('cp932')
            reader = csv.reader(io.StringIO(content))
            for r in reader:
                if len(r) >= 2: input_data.append((r[0].strip(), r[1].strip().upper().replace('-', '').replace(' ', '')))

    if input_data:
        for n, s in input_data:
            count = 0
            for i in range(len(s) - KMER_LEN + 1):
                res = evaluate_sirna_candidates(s[i:i+KMER_LEN], TM_THRESHOLD, RESTRICT_3PRIME_AU)
                if res:
                    results.append([n, i+1, s[i:i+KMER_LEN], res['g'], res['p'], res['tg'], res['tp']])
                    count += 1
            print(f"   - {n}: Found {count} candidates.")

# ==========================================
# Mode B: Common Target (with auto-numbering)
# ==========================================
else:
    sequences = {}
    if "Manual" in INPUT_METHOD:
        text_io = io.StringIO(COMMON_FASTA_TEXT.strip())
        for record in SeqIO.parse(text_io, "fasta"):
            sequences[record.id] = str(record.seq).upper().replace('-', '').replace(' ', '')
    else:
        uploaded = files.upload()
        if uploaded:
            fname = list(uploaded.keys())[0]
            raw_data = uploaded[fname]
            try: content = raw_data.decode('utf-8-sig')
            except: content = raw_data.decode('cp932')
            text_io = io.StringIO(content)
            for record in SeqIO.parse(text_io, "fasta"):
                sequences[record.id] = str(record.seq).upper().replace('-', '').replace(' ', '')

    if len(sequences) >= 2:
        seq_names = list(sequences.keys())
        print(f"🎯 Targets ({len(seq_names)}): {', '.join(seq_names)}")
        ref_name, ref_seq = seq_names[0], sequences[seq_names[0]]
        other_seqs = [sequences[name] for name in seq_names[1:]]
        checked_kmers = set()
        common_id_counter = 1

        for i in range(len(ref_seq) - KMER_LEN + 1):
            kmer = ref_seq[i:i+KMER_LEN]
            if kmer in checked_kmers: continue
            checked_kmers.add(kmer)

            if all(kmer in o_seq for o_seq in other_seqs):
                res = evaluate_sirna_candidates(kmer, TM_THRESHOLD, RESTRICT_3PRIME_AU)
                if res:
                    # Automatic Numbering: Common_Target_1, Common_Target_2...
                    results.append([f"Common_Target_{common_id_counter}", f"{ref_name}:{i+1}", kmer, res['g'], res['p'], res['tg'], res['tp']])
                    common_id_counter += 1

# --- 4. Output ---
if results:
    df = pd.DataFrame(results, columns=['Target_Name', 'Position(5to3)', 'Target_DNA(23bp)', 'Guide_RNA', 'Passenger_RNA', 'Tm_Guide', 'Tm_Passenger'])
    print(f"\n✅ Success! {len(df)} candidates found.")
    display(df.head(10))
    df.to_csv(out_fname, index=False, encoding='utf-8-sig')
    files.download(out_fname)
else:
    print("\n❌ No candidates found.")

In [ ]:
#@title Step 3: Genome Database Setup
#@markdown Pre-process the genome for specificity checking. Using Google Drive is highly recommended for large mammalian genomes.

#@markdown ---
#@markdown #### **1. Data Source Selection**
GENOME_SOURCE = "Google Drive (Recommended)" #@param ["Google Drive (Recommended)", "Auto-download from NCBI", "Upload Custom FASTA"]

#@markdown ---
#@markdown ### **▼ IF [Google Drive] OR [NCBI] IS SELECTED ▼**
#@markdown *(Ignore this section if you selected "Upload Custom FASTA")*
#@markdown - **Google Drive:** Enter the exact file path (e.g., `/content/drive/MyDrive/genome.fna`)
#@markdown - **NCBI:** Enter the Accession number (e.g., `GCF_947179515.1`)
INPUT_VALUE = "" #@param {type:"string"}

import os, subprocess, glob, sys, time
from google.colab import files, drive

print(f"🚀 Running Step 3: Database Construction")

# --- 1. Mount Google Drive if needed ---
if "Drive" in GENOME_SOURCE:
    if not os.path.exists("/content/drive"):
        print("📂 Mounting Google Drive. Please follow the authentication prompt...")
        drive.mount('/content/drive')

# --- 2. BLAST+ Setup ---
if subprocess.run(["which", "makeblastdb"], capture_output=True).returncode != 0:
    print("📦 Installing BLAST+ toolkit... (Silent install)")
    !apt-get update -y > /dev/null 2>&1 && apt-get install -y ncbi-blast+ > /dev/null 2>&1

fasta_file = ""
# Clean up user input (remove accidental quotes or spaces)
input_clean = INPUT_VALUE.strip().strip("'").strip('"')

# --- 3. Handle Genome Source ---
if "Drive" in GENOME_SOURCE:
    if os.path.exists(input_clean):
        fasta_file = input_clean
        print(f"✅ Found genome on Drive: {os.path.basename(fasta_file)}")
    else:
        print(f"❌ Error: File not found at '{input_clean}'. Please check the path.")

elif "NCBI" in GENOME_SOURCE:
    if not input_clean:
        print("❌ Error: Accession number cannot be empty.")
    else:
        print(f"📥 Fetching data for '{input_clean}' from NCBI (Quick-mode: genome only)...")
        if not os.path.exists("datasets"):
            !curl -s -o datasets "https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets"
            !chmod +x datasets

        # Download strictly the genomic sequence to save time/bandwidth
        !./datasets download genome accession {input_clean} --filename dataset.zip --include genome > /dev/null 2>&1

        if os.path.exists("dataset.zip"):
            !unzip -q -o dataset.zip -d ncbi_dataset
            fna_files = glob.glob("ncbi_dataset/ncbi_dataset/data/**/*.fna", recursive=True)
            if fna_files:
                fasta_file = fna_files[0]
                print(f"✅ Successfully downloaded: {os.path.basename(fasta_file)}")
            else:
                print("❌ Error: No .fna file found in the downloaded dataset.")
        else:
            print("❌ Error: Failed to download from NCBI. Check the Accession number or server status.")

else:
    print("📂 Please upload your custom FASTA file.")
    uploaded = files.upload()
    if uploaded:
        fasta_file = list(uploaded.keys())[0]

# --- 4. Build BLAST Database ---
if fasta_file and os.path.exists(fasta_file):
    db_name = "current_project_db"
    print(f"\n🔨 Building BLAST database... ")
    subprocess.run(["makeblastdb", "-in", fasta_file, "-dbtype", "nucl", "-out", db_name], stdout=subprocess.DEVNULL)
    os.environ['ACTIVE_BLAST_DB'] = db_name

    print("\n" + "—"*50)
    print("✅ Database construction is complete!")
    print("—"*50)

In [ ]:
#@title Step 4: Genome-wide Off-target Profiling (Specificity Check)
#@markdown Upload the CSV file generated in Step 2 to evaluate off-target risks across the entire genome.

#@markdown ---
#@markdown #### **1. Analysis Settings**
#@markdown How many loci (positions) in the genome are you targeting?
#@markdown (Set to 1 for a single gene, or the number of genes for a "common" target).
EXPECTED_LOCI = 1 #@param {type:"integer"}
MAX_TARGET_SEQS = 1000 #@param {type:"integer"}

import os, subprocess, pandas as pd, io, time
from google.colab import files

# --- 1. Project and Environment Setup ---
p_name = PROJECT_NAME if 'PROJECT_NAME' in globals() else "Untitled_Project"
# Retrieve the database name created in Step 3
db_name = os.environ.get('ACTIVE_BLAST_DB', 'current_project_db')

print(f"🚀 Running Step 4: {p_name}")

# --- 2. Check Software Availability ---
if subprocess.run(["which", "blastn"], capture_output=True).returncode != 0:
    print("📦 Installing BLAST+ toolkit... (Approx. 30s)")
    !apt-get update -y > /dev/null 2>&1 && apt-get install -y ncbi-blast+ > /dev/null 2>&1

# --- 3. Check Genome Database ---
if not os.path.exists(f"{db_name}.nsq"):
    print(f"❌ Error: Genome database '{db_name}' not found. Please run Step 3 first.")
else:
    # --- 4. Upload Query File ---
    print(f"\n📂 Please upload the candidate CSV file generated in Step 2.")
    uploaded = files.upload()

    if uploaded:
        fname = list(uploaded.keys())[0]
        raw_data = uploaded[fname]

        # Robust encoding handler
        try:
            df_queries = pd.read_csv(io.BytesIO(raw_data), encoding='utf-8-sig')
        except UnicodeDecodeError:
            print("💡 Note: UTF-8 encoding failed. Retrying with Shift-JIS (cp932)...")
            df_queries = pd.read_csv(io.BytesIO(raw_data), encoding='cp932')

        # Check if necessary column exists
        if 'Target_DNA(23bp)' not in df_queries.columns:
             print("❌ Error: Column 'Target_DNA(23bp)' not found in the uploaded CSV.")
        else:
            # --- 5. Run BLAST Search ---
            print(f"\n🔍 Searching genome for potential off-targets (blastn-short)...")
            with open("temp_query.fa", "w") as f:
                for idx, row in df_queries.iterrows():
                    f.write(f">q_{idx}\n{row['Target_DNA(23bp)']}\n")

            subprocess.run([
                "blastn", "-query", "temp_query.fa", "-db", db_name,
                "-task", "blastn-short", "-outfmt", "6 qseqid nident length",
                "-max_target_seqs", str(MAX_TARGET_SEQS), "-out", "blast_res.txt"
            ])

            # --- 6. Analyze Results and Risk Profiling ---
            if os.path.exists("blast_res.txt") and os.path.getsize("blast_res.txt") > 0:
                blast_res = pd.read_csv("blast_res.txt", sep='\t', names=['qid', 'nid', 'length'])

                stats = []
                for idx in df_queries.index:
                    q_id = f"q_{idx}"
                    q_hits = blast_res[blast_res['qid'] == q_id]

                    # Total perfect matches (23/23nt)
                    total_perfect = len(q_hits[q_hits['nid'] == 23])

                    # Unexpected perfect matches (Off-targets like pseudogenes)
                    off_target_perfect = max(0, total_perfect - EXPECTED_LOCI)

                    # Count mismatch hits by risk level
                    # High Risk: 1-3 mismatches (20-22nt match)
                    high_risk = len(q_hits[(q_hits['nid'] >= 20) & (q_hits['nid'] <= 22)])
                    # Moderate Risk: Seed-region or partial matches (16-19nt match)
                    mod_risk = len(q_hits[(q_hits['nid'] >= 16) & (q_hits['nid'] <= 19)])

                    stats.append({
                        'Expected_Loci': EXPECTED_LOCI,
                        'Total_Perfect(23nt)': total_perfect,
                        'OffTarget_Perfect': off_target_perfect,
                        'High_Risk_OT(20-22nt)': high_risk,
                        'Mod_Risk_OT(16-19nt)': mod_risk
                    })

                # Merge with original data
                final_df = pd.concat([df_queries, pd.DataFrame(stats)], axis=1)

                # Sorting Logic:
                # 1. Fewer "Perfect Off-targets" (Mandatory)
                # 2. Fewer "High Risk Off-targets"
                # 3. Fewer "Moderate Risk Off-targets"
                final_df = final_df.sort_values(
                    by=['OffTarget_Perfect', 'High_Risk_OT(20-22nt)', 'Mod_Risk_OT(16-19nt)'],
                    ascending=[True, True, True]
                )

                # --- 7. Display Results and Download ---
                print(f"\n✅ Analysis Complete! Candidates are sorted by their specificity profile.")
                display(final_df.head(10))

                out_fname = f"Step4_OffTarget_Evaluated_{p_name}.csv"
                final_df.to_csv(out_fname, index=False, encoding='utf-8-sig')

                print(f"\n💾 Preparing result file '{out_fname}'...")
                time.sleep(1)
                files.download(out_fname)

                print("\n" + "—"*50)
                print("　　　✅ Step 4 Complete!")
                print("—"*50)
            else:
                print("❌ Error: No BLAST hits found. Please check your query or database.")

In [ ]:
#@title Step 5: Template DNA Oligo Generation
#@markdown This final step generates DNA template oligos tailored for T7 in vitro transcription.
#@markdown Please upload the CSV file evaluated in Step 4.

#@markdown ---
#@markdown #### **1. Promoter and Extension Settings**
#@markdown Choose a 5' extension sequence (6 bp) to enhance IVT efficiency.
EXTENSION_MODE = "Default (GGCTCG)" #@param ["Default (GGCTCG)", "Custom (Enter below)", "None"]
CUSTOM_EXTENSION = "" #@param {type:"string"}

#@markdown Choose the core promoter sequence.
PROMOTER_MODE = "Default T7 (TAATACGACTCACTATAGGG)" #@param ["Default T7 (TAATACGACTCACTATAGGG)", "Custom (Enter below)"]
CUSTOM_PROMOTER = "" #@param {type:"string"}

import pandas as pd
import io, time
from google.colab import files

# --- 1. Resolve Final Sequences ---
def get_promoter_setup():
    # Resolve Extension
    if EXTENSION_MODE == "Default (GGCTCG)":
        ext = "GGCTCG"
    elif EXTENSION_MODE == "Custom (Enter below)":
        ext = CUSTOM_EXTENSION.strip().upper()
    else:
        ext = ""

    # Resolve Promoter
    if PROMOTER_MODE == "Default T7 (TAATACGACTCACTATAGGG)":
        prom = "TAATACGACTCACTATAGGG"
    else:
        prom = CUSTOM_PROMOTER.strip().upper()

    return ext + prom

FINAL_PROMOTER_FULL = get_promoter_setup()

# --- 2. Environment Setup ---
p_name = PROJECT_NAME if 'PROJECT_NAME' in globals() else "Untitled_Project"
print(f"🚀 Running Step 5: {p_name}")
print(f"🧬 Final Template Prefix: {FINAL_PROMOTER_FULL}")

# --- 3. File Upload ---
print(f"\n📂 Please upload the specificity-checked CSV from Step 4.")
uploaded = files.upload()

if not uploaded:
    print("❌ Error: No file selected.")
else:
    fname = list(uploaded.keys())[0]
    raw_data = uploaded[fname]

    try:
        df_in = pd.read_csv(io.BytesIO(raw_data), encoding='utf-8-sig')
    except UnicodeDecodeError:
        print("💡 Note: UTF-8 encoding failed. Retrying with Shift-JIS (cp932)...")
        df_in = pd.read_csv(io.BytesIO(raw_data), encoding='cp932')

    # Column detection
    name_col = next((c for c in df_in.columns if 'Name' in c), None)
    g_rna_col = 'Guide_RNA'
    p_rna_col = 'Passenger_RNA'

    if not all([name_col, g_rna_col, p_rna_col]):
        print(f"❌ Error: Required columns missing.")
    else:
        # --- 4. Oligo Generation ---
        oligos = []
        def get_rc(seq):
            return seq.translate(str.maketrans("ATGC", "TACG"))[::-1]

        for _, row in df_in.iterrows():
            label_base = str(row[name_col])

            for strand_type, col_name in [('guide', g_rna_col), ('pass', p_rna_col)]:
                target_dna = str(row[col_name]).upper().replace('U', 'T')

                # Synthesis strand: [Extension + Promoter] + Target DNA
                sy_seq = FINAL_PROMOTER_FULL + target_dna
                # Complementary strand: Reverse complement of synthesis
                co_seq = get_rc(sy_seq)

                oligos.append({"Oligo Name": f"{label_base}:{strand_type}:sy", "Sequence": sy_seq})
                oligos.append({"Oligo Name": f"{label_base}:{strand_type}:co", "Sequence": co_seq})

        # --- 5. Final Display and Download ---
        df_out = pd.DataFrame(oligos)
        print(f"\n✅ Success! {len(df_out)} oligos generated.")
        display(df_out.head(4))

        out_fname = f"Step5_Template_Oligos_{p_name}.csv"
        df_out.to_csv(out_fname, index=False, encoding='utf-8-sig')

        print(f"\n💾 Downloading '{out_fname}'...")
        time.sleep(1)
        files.download(out_fname)

        print("\n" + "—"*50)
        print("　　　🎉 ALL PIPELINE STEPS COMPLETE! 🎉")
        print("—"*50)